In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/True.csv
/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/Fake.csv


In [2]:
fake_path = "/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/Fake.csv"
true_path = "/kaggle/input/datasets/clmentbisaillon/fake-and-real-news-dataset/True.csv"

In [3]:
fake_df = pd.read_csv(fake_path)
true_df = pd.read_csv(true_path)

In [4]:
print("size of fake df",fake_df.shape)
print("size of true df",true_df.shape)

size of fake df (23481, 4)
size of true df (21417, 4)


In [5]:
fake_df.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [6]:
true_df.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [7]:
print("\nFake News Columns:")
print(fake_df.columns.tolist())

print("\nReal News Columns:")
print(true_df.columns.tolist())


Fake News Columns:
['title', 'text', 'subject', 'date']

Real News Columns:
['title', 'text', 'subject', 'date']


In [12]:
fake_df["label"] = 0
true_df["label"] = 1

df = pd.concat([fake_df,true_df],ignore_index = True)

print(df.shape)
display(df.head())

(44898, 5)


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [14]:
#Now suffle the dataset so that the true and false news are mixed

df = df.sample(frac=1,random_state=42).reset_index(drop=True)
df.head()

,title,text,subject,date,label
0,It’s Really Happening: Trump Adviser Lays Out...,"Well, that didn t take long. In the short time...",News,"November 16, 2016",0
1,Republican attempt to deflect Trump-Russia pro...,(Reuters) - Republican lawmaker Devin Nunes’ i...,politicsNews,"September 11, 2017",1
2,Trump says churches should get FEMA funds for ...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"September 9, 2017",1
3,Trump Loves To Say The New York Times Is ‘Fai...,Print journalism and longstanding papers have ...,News,"August 7, 2017",0
4,House Speaker Ryan briefed Trump on healthcare...,WASHINGTON (Reuters) - President Donald Trump ...,politicsNews,"March 24, 2017",1


In [15]:
#examine missing values
df.isnull().sum()


title      0
text       0
subject    0
date       0
label      0
dtype: int64

In [16]:
#duplicate
df.duplicated().sum()

np.int64(209)

In [18]:
df = df.drop_duplicates().reset_index(drop=True)

df.shape

(44689, 5)

In [19]:
df.duplicated().sum()

np.int64(0)

In [21]:
#Prepare the text data
#check sample news
print("Sample Fake news")
print(df[df["label"]==0]['text'].iloc[0][:500])

print("Sample Real news")
print(df[df["label"]==1]['text'].iloc[0][:500])


print("text length statistics")
print(df["text"].str.len().describe())

Sample Fake news
Well, that didn t take long. In the short time since Americans kinda-sorta elected Donald Trump to be Pussygrabber-in-Chief, Trump has appointed a bona fide white nationalist to a high-level position and perform numerous other actions that months ago were considered extremely unlikely nightmare scenarios   and the latest seems to be that the Muslim registry the media told us Trump was not serious about might actually happen.Under the Trump regime, Muslim immigrants coming to the United States fr
Sample Real news
(Reuters) - Republican lawmaker Devin Nunes’ investigation into whether Obama administration officials used classified intelligence reports to discredit Donald Trump’s 2016 campaign team could backfire on the congressman - and the president, sources familiar with the reports said. The reports contain no evidence that any aides to former Democratic President Barack Obama acted improperly, the sources said, but they do indicate some Trump associates may have viol

In [22]:
# Find unusually short news articles
df["text_length"] = df["text"].str.len()

print("News with fewer than 20 characters")

df.loc[df["text_length"]<20,["title","text","label","text_length"]]



News with fewer than 20 characters


,title,text,label,text_length
6,DELUSIONAL OBAMA BLAMES “Fake News” For Hillar...,,0,1
34,STUNNING! FORMER DISNEY TECH WORKER’S Emotiona...,,0,1
79,DIRTY JOBS’ MIKE ROWE: Great Opportunities Out...,,0,1
174,ARROGANT ILLEGAL ALIEN Who Voted 5 Times In 20...,Watch:,0,6
420,ROB SCHNEIDER Nails The Russia Conspiracy Theo...,,0,1
...,...,...,...,...
44473,DANIEL HANNAN Tells “The Generation Of The Saf...,,0,1
44475,GENERAL FLYNN: The Right Stuff…You’re In Good ...,,0,1
44552,NOTE TO SATURDAY NIGHT LIVE: Making Fun Of Tru...,,0,1
44600,SHOCKER! CNN POLITICAL DIRECTOR On Trump’s Fir...,,0,1


In [23]:
# Check empty or whitespace-only text
empty_text = df["text"].isna() | df["text"].str.strip().eq("")

print("Empty text rows:", empty_text.sum())

Empty text rows: 631


In [24]:
# Replace missing values with empty strings
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

# Combine title and article text
df["content"] = (
    df["title"].str.strip() + " " + df["text"].str.strip()
).str.strip()

# Remove rows where combined content is empty
df = df[df["content"].ne("")].reset_index(drop=True)

print("Dataset shape:", df.shape)
print("\nSample combined content:")
print(df["content"].iloc[0][:500])

Dataset shape: (44689, 7)

Sample combined content:
It’s Really Happening: Trump Adviser Lays Out Plan For National Muslim Registry Well, that didn t take long. In the short time since Americans kinda-sorta elected Donald Trump to be Pussygrabber-in-Chief, Trump has appointed a bona fide white nationalist to a high-level position and perform numerous other actions that months ago were considered extremely unlikely nightmare scenarios   and the latest seems to be that the Muslim registry the media told us Trump was not serious about might actually


In [28]:
from sklearn.model_selection import train_test_split

X = df["content"]
y = df["label"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [29]:
len(X_train)

35751

In [30]:
len(X_test)

8938

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [35]:
tfidf = TfidfVectorizer(max_features=5000,stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
print("Training data shape:", X_train_tfidf.shape)
print("Testing data shape:", X_test_tfidf.shape)

Training data shape: (35751, 5000)
Testing data shape: (8938, 5000)


In [36]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf,y_train)

LogisticRegression(max_iter=1000)

In [37]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict on test data
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy * 100, 2), "%")

# Detailed evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Fake News", "Real News"]))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 98.74 %

Classification Report:
              precision    recall  f1-score   support

   Fake News       0.99      0.98      0.99      4696
   Real News       0.98      0.99      0.99      4242

    accuracy                           0.99      8938
   macro avg       0.99      0.99      0.99      8938
weighted avg       0.99      0.99      0.99      8938


Confusion Matrix:
[[4623   73]
 [  40 4202]]


In [39]:
def predict_news(news):
    news_tfidf = tfidf.transform([news])

    prediction = model.predict(news_tfidf)[0]
    probabilities = model.predict_proba(news_tfidf)[0]

    print("Prediction:", "Fake News" if prediction == 0 else "Real News")
    print(f"Fake News probability: {probabilities[0] * 100:.2f}%")
    print(f"Real News probability: {probabilities[1] * 100:.2f}%")

news = input("Enter news text: ")
predict_news(news)

Enter news text:  SHOCKING! Government Hiding Secret Evidence That Will Change Everything! Share This Before It Gets Deleted!


Prediction: Fake News
Fake News probability: 93.98%
Real News probability: 6.02%


In [40]:
# Create a DataFrame to compare actual and predicted labels
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "News": X_test
})

# Select incorrectly classified news
wrong_predictions = results[
    results["Actual"] != results["Predicted"]
]

print("Total incorrect predictions:", len(wrong_predictions))

# Display 10 incorrect predictions
display(wrong_predictions.head(10))

Total incorrect predictions: 113


,Actual,Predicted,News
41962,1,0,"Factbox: The nasty things Trump, Romney have s..."
11421,1,0,Thomas ends decade of self-imposed silence at ...
9034,1,0,Djokovic hoping Kyrgios can stop squandering h...
12977,0,1,VICE PRESIDENT PENCE BREAKS TIE In Bill Allowi...
22190,1,0,Argentine TV ad mocks Trump to promote soccer ...
1055,0,1,OBAMA USES LABOR DAY TO HARM PRIVATE SECTOR WI...
550,0,1,BREAKING: MASSIVE CYBER ATTACK ON ALL FEDERAL ...
21008,1,0,"Obama says Trump unfit for presidency, has 'ou..."
13814,0,1,BREAKING: A HUGE DEAL! Putin Suspends Nuclear ...
16349,0,1,BREAKING NEWS: GOP Controlled House Votes To R...


In [41]:
# Display 10 misclassified news titles and their labels
wrong_predictions["Title"] = df.loc[
    wrong_predictions.index, "title"
]

display(wrong_predictions[["Actual", "Predicted", "Title"]].head(10))

/tmp/ipykernel_48/666769799.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  wrong_predictions["Title"] = df.loc[


,Actual,Predicted,Title
41962,1,0,"Factbox: The nasty things Trump, Romney have s..."
11421,1,0,Thomas ends decade of self-imposed silence at ...
9034,1,0,Djokovic hoping Kyrgios can stop squandering h...
12977,0,1,VICE PRESIDENT PENCE BREAKS TIE In Bill Allowi...
22190,1,0,Argentine TV ad mocks Trump to promote soccer ...
1055,0,1,OBAMA USES LABOR DAY TO HARM PRIVATE SECTOR WI...
550,0,1,BREAKING: MASSIVE CYBER ATTACK ON ALL FEDERAL ...
21008,1,0,"Obama says Trump unfit for presidency, has 'ou..."
13814,0,1,BREAKING: A HUGE DEAL! Putin Suspends Nuclear ...
16349,0,1,BREAKING NEWS: GOP Controlled House Votes To R...


In [43]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Prepare headline data
X_title = df["title"].fillna("")
y = df["label"]

# Split the data
X_train_title, X_test_title, y_train_title, y_test_title = train_test_split(
    X_title,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Convert headlines into TF-IDF features
title_tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_title_tfidf = title_tfidf.fit_transform(X_train_title)
X_test_title_tfidf = title_tfidf.transform(X_test_title)

# Train the model
title_model = LogisticRegression(max_iter=1000)
title_model.fit(X_train_title_tfidf, y_train_title)

# Evaluate the model
title_pred = title_model.predict(X_test_title_tfidf)

print("Headline Model Accuracy:",
      round(accuracy_score(y_test_title, title_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(
    y_test_title,
    title_pred,
    target_names=["Fake News", "Real News"]
))

Headline Model Accuracy: 94.42 %

Classification Report:
              precision    recall  f1-score   support

   Fake News       0.95      0.94      0.95      4696
   Real News       0.94      0.95      0.94      4242

    accuracy                           0.94      8938
   macro avg       0.94      0.94      0.94      8938
weighted avg       0.94      0.94      0.94      8938



In [44]:
import joblib

# Save the trained model and TF-IDF vectorizer
joblib.dump(model, "/kaggle/working/fake_news_model.pkl")
joblib.dump(tfidf, "/kaggle/working/tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [45]:
import os

print(os.listdir("/kaggle/working"))

['fake_news_model.pkl', 'tfidf_vectorizer.pkl', '.virtual_documents']


In [46]:
import joblib

model = joblib.load("/kaggle/working/fake_news_model.pkl")
tfidf = joblib.load("/kaggle/working/tfidf_vectorizer.pkl")

news = ["Scientists announce a new breakthrough in renewable energy technology."]

news_tfidf = tfidf.transform(news)
prediction = model.predict(news_tfidf)[0]

if prediction == 0:
    print("Prediction: Fake News")
else:
    print("Prediction: Real News")

Prediction: Real News


In [47]:
news = ["Scientists announce a new breakthrough in renewable energy technology."]

news_tfidf = tfidf.transform(news)

prediction = model.predict(news_tfidf)[0]
probabilities = model.predict_proba(news_tfidf)[0]

print("Prediction:", "Real News" if prediction == 1 else "Fake News")
print(f"Fake News Probability: {probabilities[0] * 100:.2f}%")
print(f"Real News Probability: {probabilities[1] * 100:.2f}%")

Prediction: Real News
Fake News Probability: 49.25%
Real News Probability: 50.75%


In [48]:
test_news = [
    "BREAKING: Shocking discovery that will change everything!",
    "The government announced a new education policy on Monday.",
    "Scientists publish a study on renewable energy technology."
]

test_tfidf = tfidf.transform(test_news)
predictions = model.predict(test_tfidf)
probabilities = model.predict_proba(test_tfidf)

for news, prediction, prob in zip(
    test_news, predictions, probabilities
):
    print("\nNews:", news)
    print("Prediction:", "Real News" if prediction == 1 else "Fake News")
    print(f"Fake Probability: {prob[0] * 100:.2f}%")
    print(f"Real Probability: {prob[1] * 100:.2f}%")


News: BREAKING: Shocking discovery that will change everything!
Prediction: Fake News
Fake Probability: 99.07%
Real Probability: 0.93%

News: The government announced a new education policy on Monday.
Prediction: Real News
Fake Probability: 34.89%
Real Probability: 65.11%

News: Scientists publish a study on renewable energy technology.
Prediction: Fake News
Fake Probability: 53.52%
Real Probability: 46.48%


In [49]:
from IPython.display import FileLink, display

display(FileLink("/kaggle/working/fake_news_model.pkl"))
display(FileLink("/kaggle/working/tfidf_vectorizer.pkl"))

/kaggle/working/fake_news_model.pkl

/kaggle/working/tfidf_vectorizer.pkl